# Text Mining II: routing messages automatically
## The help desk wants a model that sends each message to the right team
**Salem Othman · Student notebook**

On Wednesday the help-desk messages had no labels, so you used clustering to find topics. Since then the help desk has given us **570 messages from last semester**, each **tagged** by a staff member with the team that fixed it: wifi, printing, heating, dining or laundry.

With labels, this becomes **classification**: learn from tagged messages, then predict the team for a new one. You will build **Naive Bayes** by hand on 5 short messages, train it on the real messages, find out where and why it is wrong, look inside it, and decide which messages it should route on its own and which should go to a person.

### Today's five steps
| Part | Step | What you do | Exercise |
|---|---|---|---|
| 1 | By hand | Run Naive Bayes on 5 short messages, then check it with scikit-learn | E1 |
| 2 | Train | Split the tagged messages, train a router, compare with the baseline | E2 |
| 3 | Errors | Read the confusion matrix and the messages it got wrong | E3 |
| 4 | Inside | See which words the model relies on, and test your own messages | E4 |
| 5 | Route | Decide which messages the model routes and which go to a person | E5 |

**For every exercise:** predict → write the code → check → explain.

**How to work**
- Run cells from top to bottom with **Shift + Enter**.
- In an exercise cell, **write the code yourself** under each numbered step. Keep the variable names exactly as written; the Check cell looks for them.
- `PASS` means that step is done. `FIX` tells you what to look at.
- If a Check cell says an exercise **is not finished yet**, that is not an error: write the code in the exercise cell above, run it, then run the Check cell again.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 160)
RANDOM_STATE = 42

def check(label, passed):
    """Print PASS or FIX for one requirement."""
    print(("PASS  " if passed else "FIX   ") + label)

def ready(value, step):
    """True when the exercise above is done. Otherwise print a reminder and return False.

    It never raises, so a blank exercise cell can never stop the class with a red
    traceback: the Check cell just tells the student what to do next.
    """
    if value is None:
        print(f"{step} is not finished yet.")
        print("Write your code in the exercise cell above, run that cell, then run this one again.")
        return False
    return True

print("Setup complete.")

Setup complete.


# Part 0 · The tagged messages
| Column | Meaning |
|---|---|
| `message_id` | Ticket number. Never a model input. |
| `message` | What the student wrote |
| `team` | **The answer:** the team that fixed it (wifi, printing, heating, dining, laundry) |

In [2]:
rng = np.random.default_rng(23)

# Last semester's help-desk messages. A staff member read each one and tagged the team
# that fixed it: wifi, printing, heating, dining or laundry.
greetings = ["", "", "", "Hi, ", "Hello, ", "Hey, "]
places = ["in my room", "in Baker Hall", "on the 3rd floor of Rodgers", "in the library",
          "in the Annex", "in Evans Way dorm", "in my suite", "near the study lounge"]
times = ["since this morning", "again today", "every night this week", "for two days",
         "since Monday", "all weekend", ""]
asks = ["Please fix it.", "Can someone help?", "Thanks.", "This keeps happening.",
        "Who should I contact?", "Please send someone.", ""]

problems = {
    "wifi": ["the wifi keeps disconnecting", "the wifi is really slow", "I can't connect to the wifi",
             "the internet connection drops", "the network signal is weak", "eduroam won't let me log in",
             "the wifi router is blinking red", "my laptop loses the wireless connection"],
    "printing": ["the printer is jammed", "the printer is out of paper", "my print job is stuck in the queue",
                 "the printer says low toner", "my print credits were charged but nothing printed",
                 "color printing does not work", "the printer prints blank pages", "the scanner on the printer is broken"],
    "heating": ["the heat is not working", "the radiator is cold", "my room is freezing",
                "the thermostat is stuck on high", "it is way too hot and the heating won't turn off",
                "the heater makes a loud banging noise", "the air conditioning is blowing warm air",
                "the window does not close and the room is cold"],
    "dining": ["the dining hall food was cold", "the dining hall ran out of vegetarian options at dinner",
               "the lunch line is extremely long", "my meal swipe was charged twice",
               "the salad bar was empty at lunch", "the pizza at dinner was undercooked",
               "the dining hall menu online is wrong", "the dining hall closed early before dinner"],
    "laundry": ["the washing machine won't start", "the dryer takes my money but does not dry",
                "the laundry card reader is broken", "the washer leaked water on the floor",
                "my clothes came out of the dryer still wet", "two laundry machines are out of order",
                "the laundry room door is locked", "the washer stopped in the middle of the cycle"],
}
# A second sentence that adds detail about the same problem.
details = {
    "wifi": ["The wifi is unusable for my online classes.", "I already restarted the router.",
             "Other students have the same wifi problem.", "I need the internet for my online exam."],
    "printing": ["I need to print my lab report.", "I tried another printer and it did the same thing.",
                 "The printer screen shows an error.", "My assignment is due and I have to print it."],
    "heating": ["It is freezing at night.", "I had to sleep in a jacket.",
                "The heating was fine last week.", "Can maintenance check the heating?"],
    "dining": ["The food was not ready on time.", "I paid with my meal plan.",
               "Many students skipped dinner.", "The dining staff said they could not help."],
    "laundry": ["I lost two dollars in the machine.", "My clothes are still stuck in the washer.",
                "The laundry machines need repair.", "I have no clean clothes for tomorrow."],
}
# Half the time the second sentence says nothing about the topic.
general_details = ["I have an exam tomorrow.", "This is the third time this month.", "I already told the front desk.",
                   "It was fine yesterday.", "My roommate has the same problem.", "I paid for this with my card.",
                   "Nobody answered the phone.", "It is really frustrating."]
# Phrases that use another topic's words, the way real students write.
problems["printing"] += ["the machine took my card and printed nothing", "my card was charged twice for printing"]
problems["laundry"] += ["the machine took my card and did nothing", "my card was charged twice for the laundry"]
problems["dining"] += ["my card was charged twice for lunch", "it is too cold to eat in the dining area"]
problems["heating"] += ["my room is cold and the window is broken", "there is no hot water in my room"]
problems["wifi"] += ["my room has no signal", "the connection is broken again in my room"]
# Dining and laundry happen in shared places, so they get their own locations.
topic_places = {"dining": ["at the main dining hall", "at Beatty Hall", "at the cafe", "at the food court"],
                "laundry": ["in the Baker Hall laundry room", "in the basement laundry room",
                            "in the Rodgers laundry room", "on the 1st floor"]}

def write_message(topic):
    # One message in ten is sent from a place that belongs to another topic
    # (a printer in the dining hall, a wifi problem in the laundry room).
    if rng.random() < 0.15:
        where = rng.choice(topic_places["dining"] + topic_places["laundry"] + places)
    else:
        where = rng.choice(topic_places.get(topic, places))
    text = f"{rng.choice(greetings)}{rng.choice(problems[topic])} {where} {rng.choice(times)}. {rng.choice(details[topic] if rng.random() < 0.5 else general_details)} {rng.choice(asks)}"
    text = " ".join(text.split()).replace(" .", ".")
    return text[0].upper() + text[1:]

vague = ["It is not working again {where}. Please fix it.", "Nothing works {where} since Monday. Who should I contact?",
         "Same problem as last week {where}. Please send someone.", "Something is broken {where} again today. Thanks."]

rows = []
for topic, n in [("wifi", 150), ("printing", 115), ("heating", 130), ("dining", 95), ("laundry", 80)]:
    for _ in range(n):
        roll = rng.random()
        if roll < 0.10:      # two problems in one message: tagged with the first one
            other = rng.choice([t for t in problems if t != topic])
            text = f"{write_message(topic)} Also, {rng.choice(problems[other])}."
        elif roll < 0.17:    # too vague to tell; staff tagged it after phoning the student
            text = rng.choice(vague).format(where=rng.choice(places))
        else:
            text = write_message(topic)
        rows.append({"message": text, "team": topic})

tagged = pd.DataFrame(rows).sample(frac=1, random_state=4).reset_index(drop=True)
tagged.insert(0, "message_id", [f"T{i:03d}" for i in range(1, len(tagged) + 1)])

print("Tagged messages:", len(tagged))
display(tagged["team"].value_counts())
display(tagged.head(6))

Tagged messages: 570


team
wifi        150
heating     130
printing    115
dining       95
laundry      80
Name: count, dtype: int64

,message_id,message,team
0,T001,"The dining hall closed early before dinner at the cafe. My roommate has the same problem. This keeps happening. Also, the washer stopped in the middle of th...",dining
1,T002,The heater makes a loud banging noise in the library for two days. It is really frustrating. Please fix it.,heating
2,T003,"Hello, my room is cold and the window is broken in my room every night this week. It was fine yesterday. Who should I contact?",heating
3,T004,"Hello, the salad bar was empty at lunch at the main dining hall. The food was not ready on time. Thanks.",dining
4,T005,"Hey, it is way too hot and the heating won't turn off on the 3rd floor of Rodgers every night this week. I already told the front desk. Can someone help?",heating
5,T006,Color printing does not work in my suite since this morning. I already told the front desk. This keeps happening.,printing


**Write:** read the 6 messages above. Is there one you would find hard to route yourself? Why?

*Your answer:*
I would find T001 difficult, there are multiple issues that the student listed. 

# Part 1 · Naive Bayes by hand
Naive Bayes answers: *given these words, which team is most likely?* For each team it multiplies two things:

1. **The prior:** how common the team is in the training messages.
2. **The chance of each word** in the new message, for messages of that team.

The team with the biggest product wins. Then the products are rescaled so they add up to 1, which gives a probability for each team.

**Add one to every count.** If a word never appeared with a team, its chance would be 0, and one zero wipes out the whole product. So we pretend every word was seen once more than it was (**add-one smoothing**; in scikit-learn this is `alpha=1`).

**Why "naive"?** It treats the words as if each one gave independent evidence. That is false ("printer" and "toner" go together), and it works well anyway.

In [3]:
tiny_messages = ["wifi slow", "wifi router broken", "slow internet", "printer broken", "printer paper jammed"]
tiny_teams = ["wifi", "wifi", "wifi", "printing", "printing"]
new_message = "printer slow"
display(pd.DataFrame({"message": tiny_messages, "team": tiny_teams}))

,message,team
0,wifi slow,wifi
1,wifi router broken,wifi
2,slow internet,wifi
3,printer broken,printing
4,printer paper jammed,printing


### E1 · Check the hand calculation with scikit-learn
The slides worked out that "printer slow" is **wifi with 0.53, printing with 0.47**. Make scikit-learn do the same.

**Predict:** "printer" is a printing word. Why might wifi still win?
Because they both have the word broken.

In [4]:
tiny_vectorizer = None
tiny_model = None
tiny_proba = None

# Write your code below. Your code must create the variables set to None above.

# 1. tiny_vectorizer: CountVectorizer() (no stop-word list needed for these words).
#    Learn the vocabulary and count tiny_messages.
#    Hint: tiny_counts = tiny_vectorizer.fit_transform(tiny_messages)
tiny_vectorizer = CountVectorizer()
tiny_counts = tiny_vectorizer.fit_transform(tiny_messages)

# 2. tiny_model: MultinomialNB(alpha=1) fitted on the counts and tiny_teams.
tiny_model = MultinomialNB(alpha=1).fit(tiny_counts, tiny_teams)

# 3. tiny_proba: the probability of each team for new_message, as a pandas Series
#    indexed by team. Display it.
#    Hint: tiny_model.predict_proba(tiny_vectorizer.transform([new_message]))[0]
#    Hint: pd.Series(..., index=tiny_model.classes_)
tiny_proba = pd.Series(
    tiny_model.predict_proba(tiny_vectorizer.transform([new_message]))[0],
    index=tiny_model.classes_,
)

display(tiny_proba)


printing    0.470219
wifi        0.529781
dtype: float64

**Check E1**

In [5]:
if ready(tiny_model, "E1") and ready(tiny_proba, "E1"):
    # The hand calculation from the slides, done again here.
    wifi_score = 3 / 5 * (0 + 1) / (7 + 8) * (2 + 1) / (7 + 8)
    printing_score = 2 / 5 * (2 + 1) / (5 + 8) * (0 + 1) / (5 + 8)
    hand_wifi = wifi_score / (wifi_score + printing_score)
    check("tiny_proba has one probability per team", len(tiny_proba) == 2)
    check("the probabilities add up to 1", np.isclose(tiny_proba.sum(), 1))
    check("the model used add-one smoothing (alpha=1)", tiny_model.alpha == 1)
    check("scikit-learn agrees with the hand calculation", np.isclose(tiny_proba["wifi"], hand_wifi))

PASS  tiny_proba has one probability per team
PASS  the probabilities add up to 1
PASS  the model used add-one smoothing (alpha=1)
PASS  scikit-learn agrees with the hand calculation


**Write:** "printer" is a printing word, so why did wifi win? What would have happened without adding one?

*Your answer:*
Wifi won because slow is a wifi word. It appears more often. 

# Part 2 · Train and test the router
Same rules as every classifier this semester:
- **Split first**, with `stratify` so every team keeps its share in both parts.
- **Fit everything on the training messages only.** That includes the vectorizer: the vocabulary must come from the training messages, as it would in real life, where tomorrow's messages don't exist yet. A **pipeline** does this for you: `make_pipeline(CountVectorizer(...), MultinomialNB())` fits both steps on whatever you pass to `.fit`.
- **Compare with the baseline:** always predicting the most common team.

### E2 · Train the router
**Predict:** the baseline is around 0.27. What accuracy do you expect from Naive Bayes?
I expect 0.9

In [6]:
X_train = X_test = y_train = y_test = None
router = None
test_pred = None
test_accuracy = None
baseline_accuracy = None

# Write your code below. Your code must create the variables set to None above.

# 1. Split tagged["message"] and tagged["team"]: test_size=0.25, stratify by team,
#    random_state=RANDOM_STATE.
#    Hint: X_train, X_test, y_train, y_test = train_test_split(...)
X_train, X_test, y_train, y_test = train_test_split(tagged["message"], tagged['team'], test_size=0.25, random_state=RANDOM_STATE, stratify=tagged["team"])

# 2. router: a pipeline of CountVectorizer(stop_words="english") and MultinomialNB(),
#    fitted on the training messages only.
#    Hint: make_pipeline(..., ...).fit(X_train, y_train)
router = make_pipeline(CountVectorizer(stop_words="english"), MultinomialNB()).fit(X_train, y_train)

# 3. test_pred: the router's team for every test message.
#    test_accuracy: accuracy on the test messages.
test_pred = router.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)

# 4. baseline_accuracy: the share of the most common team among the test messages.
#    Hint: y_test.value_counts(normalize=True).max()
#    Print both accuracies.
baseline_accuracy = y_test.value_counts(normalize=True).max()

print(f"test_accuracy: {test_accuracy}\nbaseline_accuracy: {baseline_accuracy}")

test_accuracy: 0.958041958041958
baseline_accuracy: 0.26573426573426573


**Check E2**

In [7]:
if ready(router, "E2") and ready(test_pred, "E2") and ready(test_accuracy, "E2") and ready(baseline_accuracy, "E2"):
    vocab_train = CountVectorizer(stop_words="english").fit(X_train).vocabulary_
    check("a quarter of the messages are in the test set", abs(len(X_test) - 0.25 * len(tagged)) <= 1)
    check("the split kept each team's share (stratify)",
          np.allclose(y_test.value_counts(normalize=True).sort_index(),
                      tagged["team"].value_counts(normalize=True).sort_index(), atol=0.01))
    check("the vocabulary came from the training messages only",
          len(router.steps[0][1].vocabulary_) == len(vocab_train))
    check("test_accuracy matches the predictions", np.isclose(test_accuracy, (test_pred == y_test).mean()))
    check("the router beats the baseline", test_accuracy > baseline_accuracy)

PASS  a quarter of the messages are in the test set
PASS  the split kept each team's share (stratify)
PASS  the vocabulary came from the training messages only
PASS  test_accuracy matches the predictions
PASS  the router beats the baseline


**Write:** how much better than the baseline is the router? Why must the vectorizer be fitted on the training messages only?

*Your answer:*
The router is 3.58 times better than the baseline. The vectorizer must be fitted on the training messages to prevent leakage.

# Part 3 · Where is it wrong, and why?
Accuracy is one number. To fix a model, or to decide how far to trust it, look at **which** teams it confuses and **read** the messages it got wrong.

`classification_report` prints precision and recall for every team at once:
- **precision** of "heating": of the messages sent to heating, the share that really were heating
- **recall** of "heating": of the real heating messages, the share that reached heating

### E3 · The confusion matrix and the wrong messages
**Predict:** which kinds of message will the router get wrong?
Messages with no keywords

In [8]:
confusion = None
errors = None

# Write your code below. Your code must create the variables set to None above.

# 1. confusion: the confusion matrix as a DataFrame, rows = true team, columns = predicted team.
#    Hint: pd.DataFrame(confusion_matrix(y_test, test_pred, labels=router.classes_),
#                       index=router.classes_, columns=router.classes_)
#    Display it.
confusion = pd.DataFrame(confusion_matrix(y_test, test_pred, labels=router.classes_), index=router.classes_, columns=router.classes_)
display(confusion)
# 2. Print classification_report(y_test, test_pred, digits=3).
print(classification_report(y_test, test_pred, digits=3))

# 3. errors: the test messages the router got wrong, with columns message, true_team, predicted_team.
#    Hint: wrong = test_pred != y_test.values
#    Hint: pd.DataFrame({"message": X_test[wrong], "true_team": y_test[wrong], "predicted_team": test_pred[wrong]})
#    Display it.
wrong = test_pred != y_test.values

errors = pd.DataFrame({"message": X_test[wrong], "true_team": y_test[wrong], "predicted_team": test_pred[wrong]})

display(errors)

vec = router.named_steps["countvectorizer"]
nb = router.named_steps["multinomialnb"]
terms = vec.get_feature_names_out()

top_words = {
    team: list(terms[np.argsort(nb.feature_log_prob_[i])[::-1][:8]])
    for i, team in enumerate(nb.classes_)
}
print(top_words)

,dining,heating,laundry,printing,wifi
dining,23,0,0,0,1
heating,0,32,0,0,0
laundry,0,1,19,0,0
printing,2,0,0,26,1
wifi,0,1,0,0,37


              precision    recall  f1-score   support

      dining      0.920     0.958     0.939        24
     heating      0.941     1.000     0.970        32
     laundry      1.000     0.950     0.974        20
    printing      1.000     0.897     0.945        29
        wifi      0.949     0.974     0.961        38

    accuracy                          0.958       143
   macro avg      0.962     0.956     0.958       143
weighted avg      0.960     0.958     0.958       143



,message,true_team,predicted_team
253,"Hello, the scanner on the printer is broken near the study lounge again today. I have an exam tomorrow. Who should I contact?",printing,wifi
348,Same problem as last week on the 3rd floor of Rodgers. Please send someone.,dining,wifi
219,"The printer prints blank pages in the Annex since this morning. I have an exam tomorrow. Please send someone. Also, the dining hall food was cold.",printing,dining
33,"Hi, the laundry card reader is broken near the study lounge every night this week. I already told the front desk. Thanks.",laundry,heating
237,"The printer says low toner at Beatty Hall since Monday. I paid for this with my card. Who should I contact? Also, it is too cold to eat in the dining area.",printing,dining
532,Same problem as last week in Evans Way dorm. Please send someone.,wifi,heating


{np.str_('dining'): ['dining', 'hall', 'food', 'dinner', 'help', 'lunch', 'monday', 'hello'], np.str_('heating'): ['room', 'heating', 'cold', 'week', 'way', 'hot', 'monday', 'night'], np.str_('laundry'): ['laundry', 'room', 'machine', 'clothes', 'floor', 'card', 'washer', '1st'], np.str_('printing'): ['printer', 'print', 'card', 'hi', 'printing', 'contact', 'hall', 'printed'], np.str_('wifi'): ['wifi', 'room', 'keeps', 'connection', 'internet', 'signal', 'online', 'problem']}


**Check E3**

In [9]:
if ready(confusion, "E3") and ready(errors, "E3"):
    check("the confusion matrix covers every test message", confusion.to_numpy().sum() == len(y_test))
    check("the diagonal matches test_accuracy", np.isclose(np.trace(confusion.to_numpy()) / len(y_test), test_accuracy))
    check("errors has one row per wrong prediction", len(errors) == int((test_pred != y_test.values).sum()))
    check("every row in errors really is wrong", (errors["true_team"] != errors["predicted_team"]).all())

PASS  the confusion matrix covers every test message
PASS  the diagonal matches test_accuracy
PASS  errors has one row per wrong prediction
PASS  every row in errors really is wrong


**Write:** read every wrong message. Sort them into reasons. For each reason, could a better model fix it, or does it need a person?

*Your answer:*
253 and 33: words like exam, study lounge and broken carry stronger weights in wifi and heating. Fix: find a better model. 

348 and 532: There are no topic keywords, a person needs to review it. 

219 and 237: There are multiple problems reported. A better model can detect the multiple topics. 

# Part 4 · What did it learn?
Inside the pipeline, the Naive Bayes step stores the (log) chance of every word for every team in `feature_log_prob_`. The words with the biggest chance for a team are the words that model **relies on** for that team. Some will surprise you.

You can reach each step of a pipeline by name: `router.named_steps["countvectorizer"]` and `router.named_steps["multinomialnb"]`.

### E4 · Look inside, then test your own messages
**Predict:** will a team's top words all be words that describe the problem?
No, they will be the item that is broken. 

In [10]:
top_words = None
my_proba = None

my_messages = [
    "The printer in the dining hall is jammed",
    "My card was charged twice",
    "It is freezing and the wifi is down in my room",
    # My message: 
    "My wifi does not work in the success studio"
]

# Write your code below. Your code must create the variables set to None above.

# 1. top_words: a dictionary {team: list of its 8 words with the biggest feature_log_prob_}. Print it.
#    Hint: vec = router.named_steps["countvectorizer"];  nb = router.named_steps["multinomialnb"]
#    Hint: terms = vec.get_feature_names_out()
#    Hint: for i, team in enumerate(nb.classes_): list(terms[np.argsort(nb.feature_log_prob_[i])[::-1][:8]])
vec = router.named_steps["countvectorizer"]
nb = router.named_steps["multinomialnb"]

top_words = {}
for i, team in enumerate(nb.classes_):
    terms = vec.get_feature_names_out()
    top_words[team] = list(terms[np.argsort(nb.feature_log_prob_[i])[::-1][:8]])

display(top_words)
# 2. my_proba: a DataFrame with one row per message in my_messages and one column per team,
#    holding the router's probabilities. Display it, rounded to 2 decimals.
#    Hint: pd.DataFrame(router.predict_proba(my_messages), index=my_messages, columns=router.classes_)
my_proba = pd.DataFrame(router.predict_proba(my_messages), index=my_messages, columns=router.classes_)
display(my_proba)

# 3. Add one message of your own to my_messages and run the cell again.

{np.str_('dining'): ['dining',
  'hall',
  'food',
  'dinner',
  'help',
  'lunch',
  'monday',
  'hello'],
 np.str_('heating'): ['room',
  'heating',
  'cold',
  'week',
  'way',
  'hot',
  'monday',
  'night'],
 np.str_('laundry'): ['laundry',
  'room',
  'machine',
  'clothes',
  'floor',
  'card',
  'washer',
  '1st'],
 np.str_('printing'): ['printer',
  'print',
  'card',
  'hi',
  'printing',
  'contact',
  'hall',
  'printed'],
 np.str_('wifi'): ['wifi',
  'room',
  'keeps',
  'connection',
  'internet',
  'signal',
  'online',
  'problem']}

,dining,heating,laundry,printing,wifi
The printer in the dining hall is jammed,0.277037,0.005504,0.010952,0.704729,0.001777
My card was charged twice,0.101407,0.008289,0.168385,0.709062,0.012857
It is freezing and the wifi is down in my room,0.001384,0.105915,0.020534,0.005913,0.866253
My wifi does not work in the success studio,0.019724,0.016928,0.023271,0.808781,0.131295


**Check E4**

In [11]:
if ready(top_words, "E4") and ready(my_proba, "E4"):
    check("top_words has one list per team", set(top_words) == set(router.classes_))
    check("each list has 8 words from the vocabulary",
          all(len(w) == 8 and set(w) <= set(router.named_steps["countvectorizer"].get_feature_names_out())
              for w in top_words.values()))
    check("my_proba has one row per message and one column per team",
          my_proba.shape == (len(my_messages), len(router.classes_)))
    check("each row of probabilities adds up to 1", np.allclose(my_proba.sum(axis=1), 1))

PASS  top_words has one list per team
PASS  each list has 8 words from the vocabulary
PASS  my_proba has one row per message and one column per team
PASS  each row of probabilities adds up to 1


**Write:** (a) Which top words have nothing to do with the problem? Why did the model learn them? (b) Look at the dining-hall printer message. What does it show about how the model "reads"?

*Your answer:*
(a) monday, help, hi, hello, way. They were frequently showing up in most messages. 
(b) the model heavily favors printing issues over the other teams

# Part 5 · Route it, or ask a person?
The help desk doesn't need the router to handle **every** message. A good plan: route a message automatically when the model is **confident**, and send the rest to a person. That is a **threshold** on the top probability, the same idea as the refund threshold on Oct 2.

For each threshold you get a trade-off:
- **auto share:** the share of messages routed with no person
- **auto accuracy:** how often those automatic routes are right
- **to a person:** how many messages a staff member must read

### E5 · Choose a confidence threshold
**Predict:** if the router only acts when it is at least 90% sure, how many test messages go to a person?

In [12]:
confidence = None
threshold_table = None
chosen_threshold = None

thresholds = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

# Write your code below. Your code must create the variables set to None above.

# 1. confidence: for each test message, the router's highest probability.
#    Hint: router.predict_proba(X_test).max(axis=1)
confidence = router.predict_proba(X_test).max(axis=1)

# 2. threshold_table: one row per threshold with
#    threshold, auto_share, auto_accuracy, to_person.
#    Hint: auto = confidence >= t
#    Hint: auto_accuracy = (test_pred[auto] == y_test.values[auto]).mean()
#    Display it, rounded to 3 decimals.
t_table = {
    "threshold": [],
    "auto_share": [],
    "auto_accuracy": [],
    "to_person": []
}

for t in thresholds:
    auto = confidence >= t
    auto_share = auto.mean()
    auto_accuracy = (test_pred[auto] == y_test.values[auto]).mean()
    to_person = len(auto) - auto.sum()
    t_table["threshold"].append(t)
    t_table["auto_share"].append(auto_share)
    t_table["auto_accuracy"].append(auto_accuracy)
    t_table["to_person"].append(to_person)

threshold_table = pd.DataFrame(t_table)
display(threshold_table)
# 3. chosen_threshold: your choice, from the table. Explain it in the answer cell.
chosen_threshold = 0.7

,threshold,auto_share,auto_accuracy,to_person
0,0.50,0.993007,0.957746,1
1,0.60,0.972028,0.964029,4
2,0.70,0.958042,0.970803,6
3,0.80,0.930070,0.984962,10
4,0.90,0.909091,0.992308,13
5,0.95,0.874126,0.992000,18


**Check E5**

In [13]:
if ready(confidence, "E5") and ready(threshold_table, "E5") and ready(chosen_threshold, "E5"):
    check("one confidence per test message, between 0 and 1",
          len(confidence) == len(X_test) and bool(np.all((confidence > 0) & (confidence <= 1))))
    check("threshold_table has one row per threshold", len(threshold_table) == len(thresholds))
    check("a higher threshold never routes more messages automatically",
          threshold_table["auto_share"].is_monotonic_decreasing)
    check("auto_share and to_person agree",
          np.allclose(threshold_table["auto_share"] * len(X_test) + threshold_table["to_person"], len(X_test)))
    check("chosen_threshold is one of the thresholds", chosen_threshold in thresholds)

PASS  one confidence per test message, between 0 and 1
PASS  threshold_table has one row per threshold
PASS  a higher threshold never routes more messages automatically
PASS  auto_share and to_person agree
PASS  chosen_threshold is one of the thresholds


**Write:** which threshold would you give the help desk? Use the table, and say what the staff would still have to do.

*Your answer:*
I would give them a threshold of 0.7 because it has the highest auto share and auto accuracy. The staff will have to read about 6 messages. 

### The AI's analysis
The help-desk manager pastes your E2 and E5 results into an AI assistant. It replies:

> "Your router is 95.8% accurate, which is excellent, so you can switch off the human check and route every message automatically. Naive Bayes probabilities are well calibrated: when it says it is 97% sure, it is right 97% of the time. And the model clearly understands the messages: greetings and place names don't affect its decisions, only the words that describe the problem."

Each claim can be checked against your notebook.

**Write:** find **three** problems in the AI's answer. For each one, give the evidence from your notebook.

*Your answer:*
1. ~4% of the messages are misclassified. Messages with no topic need to be reviewed by a human. 
2. Naive Bayes is poorly calibrated and outputs overconfident probabilites near 0 or 1. 
3. Places and greetings are treated as features so it skews the predictions. 

### Challenge tasks (optional)
1. Replace `MultinomialNB()` with `LogisticRegression(max_iter=1000)` (import it from `sklearn.linear_model`) and use `TfidfVectorizer`. Is the accuracy different? Are the probabilities less extreme?
2. Use `CountVectorizer(stop_words="english", ngram_range=(1, 2))`. Do two-word phrases like *card reader* or *dining hall* change the errors?
3. Remove the place names from the messages before training (hint: `str.replace` with a list of places). Does the router get better or worse?

# Exit ticket
Answer without running more code.
1. Naive Bayes multiplies two kinds of numbers for each team. What are they, and why do we add one to every word count?
2. Why must the CountVectorizer be fitted on the training messages only? How does a pipeline help?
3. Why is it better to send low-confidence messages to a person than to route every message automatically?

*Your answers:*
1. It multiplies prior probability and likelihood. We add one to prevent unseen words having a probability of 0
2. A pipeline ensures that there is no data leakage from the test set onto the vocabulary. 
3. It is better because low confidence messages are usually multiple topics or have missing keywords. It is better to send to a human to avoid misrouting errors. 

# Submit
- Restart the kernel and run all cells. Every check should print `PASS`.
- Every *Your answer* and *Your prediction* cell is filled in.
- Submit this notebook with its outputs.